<a href="https://colab.research.google.com/github/chiragbulbule/VaultInfer/blob/main/tenseal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tenseal
import tenseal as ts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()
plain_number = [7.5]
encrypted = ts.ckks_vector(context, plain_number)
encrypted_result = encrypted * 2
result = encrypted_result.decrypt()
print(f"Original: 7.5 × 2 = 15.0")
print(f"FHE Result: {result[0]:.4f}")

In [ ]:
import tenseal as ts

# Same setup as before
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

# A vector (pretend this is a tiny AI embedding)
plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]

# Encrypt the whole vector at once
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply every element by 2, while encrypted
encrypted_result = encrypted_vector * 2

# Decrypt
result = encrypted_result.decrypt()

print("Original:  ", plain_vector)
print("Expected:  ", [x * 2 for x in plain_vector])
print("FHE Result:", [round(x, 4) for x in result])

In [ ]:
import tenseal as ts

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
poly_modulus_degree=16384,
coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]  # 6 middle levels
)
context.global_scale = 2**40
context.generate_galois_keys()

plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply but rescale between each operation
result = encrypted_vector * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2

decrypted = result.decrypt()

print("Original:       ", plain_vector)
print("Expected (×32): ", [x * 32 for x in plain_vector])
print("FHE Result:     ", [round(x, 4) for x in decrypted])

In [ ]:
import tenseal as ts

# ============================================================
# SETUP (Cryptographer's job — your code)
# ============================================================
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]
)
context.global_scale = 2**40        # ← this line is missing in your cell
context.generate_galois_keys()
# ============================================================
# CLIENT SIDE (User's machine)
# ============================================================

# Step 1: User types a query
user_query = "The machine is overheating"
print(f"User types: '{user_query}'")

# Step 2: Convert to numbers (pretend embedding, AI engineer's job)
# In real project this would be a 768-dimension vector from a real model
# For now we simulate it with small numbers
plain_embedding = [0.23, -0.87, 0.45, 1.02, -0.33]
print(f"Converted to vector: {plain_embedding}")

# Step 3: Encrypt it (YOUR job)
encrypted_embedding = ts.ckks_vector(context, plain_embedding)
print(f"\nEncrypted (what server sees): {encrypted_embedding}")

# ============================================================
# SERVER SIDE (Blind — never sees real data)
# ============================================================
print("\n--- SERVER RECEIVES ENCRYPTED BLOB ---")
print("Server has NO idea what the original query was.")

# Step 4: Server does AI math on encrypted data
# In real project this would be encrypted matrix multiplication
# For now we simulate a simple weighted sum (like a tiny neural network layer)
weights = [0.5, 0.3, 0.8, 0.2, 0.6]  # pretend model weights
encrypted_result = encrypted_embedding.dot(weights)
print("Server computed result on ciphertext — still blind!")

# ============================================================
# BACK TO CLIENT SIDE
# ============================================================
print("\n--- RESULT RETURNS TO USER ---")

# Step 5: User decrypts
decrypted_result = encrypted_result.decrypt()
score = decrypted_result[0]
print(f"Decrypted score: {round(score, 4)}")

# Step 6: Interpret result (researcher/AI engineer's job)
if score > 0:
    print("Classification: ⚠️  ALERT — Negative signal detected")
else:
    print("Classification: ✅  OK — Normal signal")

In [ ]:
!pip install tenseal
import torch
import tenseal as ts
import pandas as pd
import random
from time import time

# those are optional and are not necessary for training
import numpy as np
import matplotlib.pyplot as plt

torch.random.manual_seed(73)
random.seed(73)


def split_train_test(x, y, test_ratio=0.3):
    idxs = [i for i in range(len(x))]
    random.shuffle(idxs)
    # delimiter between test and train data
    delim = int(len(x) * test_ratio)
    test_idxs, train_idxs = idxs[:delim], idxs[delim:]
    return x[train_idxs], y[train_idxs], x[test_idxs], y[test_idxs]

def random_data(m=1024, n=2):
    # data separable by the line `y = x`
    x_train = torch.randn(m, n)
    x_test = torch.randn(m // 2, n)
    y_train = (x_train[:, 0] >= x_train[:, 1]).float().unsqueeze(0).t()
    y_test = (x_test[:, 0] >= x_test[:, 1]).float().unsqueeze(0).t()
    return x_train, y_train, x_test, y_test

def heart_disease_data():
    data = pd.read_csv("./data/framingham.csv")
    # drop rows with missing values
    data = data.dropna()

    # drop some features
    data = data.drop(columns=["education", "id"])  # , "is_male" ??

    # binarize 'male' column
    data.male = data.male.apply(lambda x: 0 if x == "F" else 1)
    data = data.rename(columns={"male": "is_male"})

    # standarize data
    for col in data.columns:
        if col != "TenYearCHD":
            data[col] = data[col] - data[col].mean()
            data[col] = data[col] / data[col].std()

    # split into train/test
    data_x = torch.tensor(data.drop(columns=["TenYearCHD"]).values).float()
    data_y = torch.tensor(data["TenYearCHD"].values).float().unsqueeze(0).t()

    x_train, y_train, x_test, y_test = split_train_test(data_x, data_y)
    return x_train, y_train, x_test, y_test

# You can use whatever data you want without modification to the tutorial
x_train, y_train, x_test, y_test = random_data()
# x_train, y_train, x_test, y_test = heart_disease_data()

print("############# Data summary #############")
print(f"x_train has shape: {x_train.shape}")
print(f"y_train has shape: {y_train.shape}")
print(f"x_test has shape: {x_test.shape}")
print(f"y_test has shape: {y_test.shape}")
print("#######################################")

#logistic regression without encryption
class LR(torch.nn.Module):

    def __init__(self, n_features):
        super(LR, self).__init__()
        self.lr = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        out = torch.sigmoid(self.lr(x))
        return out

n_features = x_train.shape[1]
model = LR(n_features)
# use gradient descent with a learning_rate=1
optim = torch.optim.SGD(model.parameters(), lr=1)
# use Binary Cross Entropy Loss
criterion = torch.nn.BCELoss()
# define the number of epochs for both plain and encrypted training
EPOCHS = 5

def train(model, optim, criterion, x, y, epochs=EPOCHS):
    for e in range(1, epochs + 1):
        optim.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optim.step()
        print(f"Loss at epoch {e}: {loss.data}")
    return model

model = train(model, optim, criterion, x_train, y_train)


In [ ]:
def accuracy(model, x, y):
    out = model(x)
    correct = torch.abs(y - out) < 0.5
    return correct.float().mean()

plain_accuracy = accuracy(model, x_test, y_test)
print(f"Accuracy on plain test_set: {plain_accuracy}")

In [ ]:
#encrypted logistic regression
class EncryptedLR:

    def __init__(self, torch_lr):
        # TenSEAL processes lists and not torch tensors,
        # so we take out the parameters from the PyTorch model
        self.weight = torch_lr.lr.weight.data.tolist()[0]
        self.bias = torch_lr.lr.bias.data.tolist()

    def forward(self, enc_x):
        # We don't need to perform sigmoid as this model
        # will only be used for evaluation, and the label
        # can be deduced without applying sigmoid
        enc_out = enc_x.dot(self.weight) + self.bias
        return enc_out

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    ################################################
    ## You can use the functions below to perform ##
    ## the evaluation with an encrypted model     ##
    ################################################

    def encrypt(self, context):
        self.weight = ts.ckks_vector(context, self.weight)
        self.bias = ts.ckks_vector(context, self.bias)

    def decrypt(self, context):
        self.weight = self.weight.decrypt()
        self.bias = self.bias.decrypt()


eelr = EncryptedLR(model)

In [ ]:
# parameters
poly_mod_degree = 4096
coeff_mod_bit_sizes = [40, 20, 40]
# create TenSEALContext
ctx_eval = ts.context(ts.SCHEME_TYPE.CKKS, poly_mod_degree, -1, coeff_mod_bit_sizes)
# scale of ciphertext to use
ctx_eval.global_scale = 2 ** 20
# this key is needed for doing dot-product operations
ctx_eval.generate_galois_keys()

In [ ]:
t_start = time()
enc_x_test = [ts.ckks_vector(ctx_eval, x.tolist()) for x in x_test]
t_end = time()
print(f"Encryption of the test-set took {int(t_end - t_start)} seconds")

In [ ]:
# (optional) encrypt the model's parameters
eelr.encrypt(ctx_eval)

In [ ]:
def encrypted_evaluation(model, enc_x_test, y_test):
    t_start = time()

    correct = 0
    for enc_x, y in zip(enc_x_test, y_test):
        # encrypted evaluation
        enc_out = model(enc_x)
        # plain comparison
        out = enc_out.decrypt()
        out = torch.tensor(out)
        out = torch.sigmoid(out)
        if torch.abs(out - y) < 0.5:
            correct += 1

    t_end = time()
    print(f"Evaluated test_set of {len(x_test)} entries in {int(t_end - t_start)} seconds")
    print(f"Accuracy: {correct}/{len(x_test)} = {correct / len(x_test)}")
    return correct / len(x_test)


encrypted_accuracy = encrypted_evaluation(eelr, enc_x_test, y_test)
diff_accuracy = plain_accuracy - encrypted_accuracy
print(f"Difference between plain and encrypted accuracies: {diff_accuracy}")
if diff_accuracy < 0:
    print("Oh! We got a better accuracy on the encrypted test-set! The noise was on our side...")

In [ ]:
import tenseal as ts

def create_context():
    # Enough depth for a small classifier (3-4 multiplications)
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        plain_modulus=-1, # Added this line
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

def encrypt_vector(context, plain_vector):
    return ts.ckks_vector(context, plain_vector)

def decrypt_vector(encrypted_vector):
    return encrypted_vector.decrypt()


# ---- TEST IT ----
context = create_context()

# Simulate a fake embedding vector (384 numbers between -1 and 1)
import random
fake_embedding = [random.uniform(-1, 1) for _ in range(384)]

# Encrypt it
encrypted = encrypt_vector(context, fake_embedding)

# Decrypt it
decrypted = decrypt_vector(encrypted)

# Check accuracy
errors = [abs(fake_embedding[i] - decrypted[i]) for i in range(384)]
print(f"Max approximation error: {max(errors):.10f}")
print(f"Average error: {sum(errors)/len(errors):.10f}")
print(f"Encryption works on 384-dim vector: ✅")

In [ ]:
# Given a list of 5 numbers, encrypt them, compute their average while encrypted
# decrypt and verify

import tenseal as ts

def create_context():
  context=ts.context(
      ts.SCHEME_TYPE.CKKS,
      poly_modulus_degree=8192,
      coeff_mod_bit_sizes=[60,40,60]
  )
  context.global_scale=2**40
vector=[10,20,30,40,50]
generate.galois_keys()
encrypted=ts.ckks_vector(context,vector)
print(encrypted)
decrypted=encrypted.decrypt()
print(decrypted)


### Resolving `ModuleNotFoundError: No module named 'sentence_embedding'`

This error typically occurs because a Python file named `sentence_embedding.py` (which contains the `embedding` and `user_embed` variables) is expected but not found in the current environment. If this is a custom file, you need to create it or upload it to your Colab instance.

I will create a dummy `sentence_embedding.py` file with empty lists for `embedding` and `user_embed` to allow the notebook to run without the `ModuleNotFoundError`. **You should replace the content of this file with your actual `sentence_embedding.py` if you have one.**

If `sentence_embedding` is meant to be an installable package, you would typically use `!pip install sentence_embedding` (or whatever the correct package name is), but this particular name often suggests a local utility file.

In [ ]:
%%writefile sentence_embedding.py

# This is a placeholder file for sentence_embedding.py
# You should replace its content with your actual code if you have it.

# Example placeholder for embedding and user_embed (assuming they are lists/arrays)
embedding = []
user_embed = []

# If your actual file has functions or classes, you would define them here:
# def some_function():
#     pass

Now that the `sentence_embedding.py` file exists, we need to create a placeholder for `dataset.py` as well, since it's also imported and likely missing. I'll create a dummy `dataset.py` with an empty `train_labels` list.

In [ ]:
%%writefile dataset.py

# This is a placeholder file for dataset.py
# You should replace its content with your actual code if you have it.

# Example placeholder for train_labels (assuming it's a list)
train_labels = []

# If your actual file has other data or functions, you would define them here:
# def load_data():
#     pass

In [ ]:
import tenseal as ts
context=ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60,40,40,60]
)
context.global_scale=2**40
context.generate_galois_keys()
vector1=[]
vector2=[]
for i in range(0,10):
  vector1.append(i)
  vector2.append(i-5)
encrypt=ts.ckks_vector(context,vector1)
encrypt2=ts.ckks_vector(context,vector2)
dot=encrypt.dot(encrypt2)
print(dot)
decrypt=dot.decrypt()
print(decrypt)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold,cross_val_score,cross_val_predict
from sentence_embedding import embedding,user_embed
from dataset import train_labels
import numpy as np
import joblib as jl

"""
This files contains the code for training and testing of the logistic regression model,along with its predictions for the given sentences(vector embeddings).

"""
#----------------------------------------------------------------------------Model Training-A-----------------------------------------------------------------#

# - Earlier used method

"""
This method used train_test_split where the dataset was divided into a 80/20 split - 80 for training and 20 for testing.
This method was abandoned because the results were not conclusive and varied on test basis.

x_train,x_test,y_train,y_test=train_test_split(embedding,train_labels,test_size=0.2,train_size=0.8,random_state=67,stratify=train_labels)
clf.fit(x_train,y_train)
accuracy=clf.score(x_test,y_test)
print(accuracy)

"""

# Currently used method

clf=LogisticRegression(max_iter=1000,class_weight="balanced",C=1)

#-----------------------------------------------------------------Cross_validation using StratifiedKFold--------------------------------------------------#

"""
This technique of cross validation is used to assess the accuracy of the model by splitting the data into 5 folds,
training and testing on each sentence to accurately assess accuracy and corresponding standard deviation of the model's accuracy.

"""

sk=StratifiedKFold(shuffle=True)

accuracy_cross_fold=cross_val_score(clf,embedding,train_labels,cv=sk)
print(f"Mean Accuracy of the model is : {accuracy_cross_fold.mean():0.2f}")
print(f"Standard deviation of the accuracy is : {np.std(accuracy_cross_fold):0.4f}")

prediction_cross_fold=cross_val_predict(clf,embedding,train_labels,cv=sk)
prediction_probability_cross_fold=cross_val_predict(clf,embedding,train_labels,cv=sk,method='predict_proba')

# print(prediction_cross_fold)
# print(prediction_probability_cross_fold)

#----------------------------------------------------------------------------Model Training-B-----------------------------------------------------------------#

"""
This is the code to actually train the model with the given dataset.

"""
clf.fit(embedding,train_labels)

#----------------------------------------------------------------------------Model Predictions-----------------------------------------------------------------#

# Accuracy of the model by inputting the same training data as the testing data - not an accurate measurement of accuracy

print(clf.score(embedding,train_labels))

# User input prediction

prediction=clf.predict(user_embed.reshape(1,384))
probability=clf.predict_proba(user_embed.reshape(1,384))

print(probability)
print(prediction)

# Test cases prediction

"""
prediction=clf.predict(test_cases_embed)
probability=clf.predict_proba(test_cases_embed)

print(probability)
print(prediction)

"""

#----------------------------------------------------------------SAVING MODEL,WEIGHTS AND BIAS TO RESPECTIVE FILES------------------------------------#

"""
weights=clf.coef_ #shape-(1,384)
bias=clf.intercept_ #shape-(1,)

flattened_weights=weights.flatten() #This is to make it compatible with tenseal parameters

np.save("./VaultInfer/Sentence_Classifier_model/New/vault_weights",flattened_weights)
np.save("./VaultInfer/Sentence_Classifier_model/New/vault_bias",bias)
jl.dump(clf,"./VaultInfer/Sentence_Classifier_model/New/vault_model",0)

"""

#----------------------------------------------------------------LOADING MODEL,WEIGHTS AND BIAS FROM RESPECTIVE FILES (TEST)------------------------------------#

"""
weights=np.load("./VaultInfer/Sentence_Classifier_model/New/vault_weights.npy")
bias=np.load("./VaultInfer/Sentence_Classifier_model/New/vault_bias.npy")

model=jl.load("./VaultInfer/Sentence_Classifier_model/New/vault_model")

print(weights[:4],weights.shape,bias)

# Testing if the model is loaded correctly

prediction=model.predict(user_embed.reshape(1,384))
probability=model.predict_proba(user_embed.reshape(1,384))

print(prediction,probability)

"""

#----------------------------------------------------------------MANUAL IMPLEMENTATION OF FORWARD PASS------------------------------------------------#

# Numpy version

"""
weighted_sum=np.dot(weights,test_embed) + bias

score=(0.5) + (weighted_sum/4) - (pow(weighted_sum,3)/48) + (pow(weighted_sum,5)/480) #-[0.81565414]-5th degree polynomial

"""

# Tenseal version

# To be completed

#--------------------------------------------------------------------------TEST CODE A---------------------------------------------------------------#

"""
predictions_all = clf.predict(embedding)
print(predictions_all)
print(type(predictions_all))


print(f"Predicted ALERT: {np.sum(predictions_all == 1)}")
print(f"Predicted NORMAL: {np.sum(predictions_all == 0)}")

test_sentences = [
    "Hydraulic pressure critical, shutdown imminent",
    "The meeting is at 3pm",
    "Unauthorized access detected at the server",
    "I'm making a cup of tea"
]
predictions = clf.predict(em)
probs=clf.predict_proba(em)
print(type(probs),probs)
print(type(predictions),predictions)

for sentence, prob in zip(test_sentences, probs):
    print(f"{prob[1]:.4f} — {sentence}")

prediction=clf.predict_proba(x_test)
for pre,actual in zip(prediction,y_test):
    print(pre,actual)

"""

#--------------------------------------------------------------------------TEST CODE B---------------------------------------------------------------#

"""

weighted_sum=np.dot(test_embed,weights) + bias #error because shapes don't meet requirements

print(np.dot(weights,test_embed)) # 1D Array

score_2=1/(1+np.exp(-weighted_sum)) #This uses sigmoid function-[0.81308656]
score_3=0.5+(0.15012*weighted_sum)-(0.001593*pow(weighted_sum,3)) #-[0.71564303] -3rd degree
score_4=0.5 + 0.197 * weighted_sum - 0.004 * weighted_sum **3 #-[ 0.777 ] -3rd degree


print(score,score_2,score_3,score_4)

all_weighted_sum=np.dot(weights,embedding.T) + bias
print(f"Min value is {all_weighted_sum.min():0.2f}")
print(f"Max value is {all_weighted_sum.max():0.2f}")
print(f"Mean value is {all_weighted_sum.mean():0.2f}")

"""

#--------------------------------------------------------------------------TEST CODE C---------------------------------------------------------------#

"""
Cross_validation using StratifiedKFold

index_list=[train_labels.index(true_label) for true_label,predict_label in zip(train_labels,prediction_cross_fold) if true_label != predict_label] - incorrect because it returns first appearance

index_list=[i for i,(actual_label,predict_label) in enumerate(zip(train_labels,prediction_cross_fold)) if actual_label!=predict_label]

wrongly_guessed_pairs=[(train_sentences[index],prediction_probability_cross_fold[index][1].item()) for index in index_list]
print(wrongly_guessed_pairs)

Testing which sentences are near the margin and which are way off

for i,predict_proba_tuple in enumerate(prediction_probability_cross_fold):
    if 0.4 <= predict_proba_tuple[1] <= 0.6:
        print(f"{train_sentences[i]} - {predict_proba_tuple[1]}")

    elif abs(predict_proba_tuple[1] - train_labels[i]) > 0.6:
        print(f"{train_sentences[i]} - {predict_proba_tuple[1]}")

"""

In [ ]:
import tenseal as ts
import numpy as np

#-------- LOAD AI ENGINEER'S WEIGHTS --------#
weights = np.load("/vault_weights.npy")
bias = np.load("/vault_bias.npy")

print(f"Weights shape: {weights.shape}")
print(f"Bias shape: {bias.shape}")
print(f"Bias value: {bias}")

In [ ]:
#-------- TENSEAL CONTEXT --------#
def create_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

context = create_context()

#-------- SIMULATE USER INPUT --------#
# Fake embedding for now — replace with real sentence embedding later
user_vector = np.random.uniform(-1, 1, 384).tolist()

#-------- ENCRYPT --------#
encrypted_vector = ts.ckks_vector(context, user_vector)
print("Vector encrypted ✅")

#-------- ENCRYPTED FORWARD PASS (server side) --------#
encrypted_dot = encrypted_vector.dot(weights.tolist())
encrypted_result = encrypted_dot + bias[0]
print("Forward pass on ciphertext complete ✅")

#-------- USER DECRYPTS --------#
decrypted_result = encrypted_result.decrypt()[0]
print(f"Decrypted weighted sum: {decrypted_result:.6f}")

#-------- POLYNOMIAL SIGMOID --------#
x = decrypted_result
score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
print(f"Probability of ALERT: {score:.4f}")

#-------- PREDICTION --------#
prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"
print(f"Prediction: {prediction}")

#-------- VERIFY AGAINST PLAINTEXT --------#
plain_dot = np.dot(weights, user_vector) + bias[0]
plain_score = 0.5 + (plain_dot/4) - (plain_dot**3/48) + (plain_dot**5/480)
plain_prediction = "ALERT 🚨" if plain_score >= 0.5 else "NORMAL ✅"

print(f"\n--- Verification ---")
print(f"FHE prediction:       {prediction}")
print(f"Plaintext prediction: {plain_prediction}")
print(f"Match: {'✅' if prediction == plain_prediction else '❌'}")

In [ ]:
!pip install sentence-transformers

In [ ]:
import tenseal as ts
import numpy as np
from sentence_transformers import SentenceTransformer

#-------- LOAD WEIGHTS --------#
weights = np.load("/vault_weights.npy")
bias = np.load("/vault_bias.npy")

#-------- LOAD EMBEDDING MODEL --------#
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model loaded ✅")

#-------- TENSEAL CONTEXT --------#
def create_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

context = create_context()
print("Encryption context ready ✅")

#-------- THE ACTUAL PIPELINE --------#
def classify_encrypted(sentence):
    print(f"\nInput: '{sentence}'")

    # Step 1 — Embed
    user_vector = embedder.encode(sentence).tolist()
    print(f"Embedded into {len(user_vector)}")

    # Step 2 — Encrypt
    encrypted_vector = ts.ckks_vector(context, user_vector)
    print(f"Vector encrypted")
    # Step 3 — Encrypted forward pass (server side)
    encrypted_dot = encrypted_vector.dot(weights.tolist())
    encrypted_result = encrypted_dot + bias[0]
    print(f"Server computed on ciphertext ")

    # Step 4 — Decrypt
    decrypted_result = encrypted_result.decrypt()[0]

    # Step 5 — Sigmoid
    x = decrypted_result
    score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
    score = max(0.0, min(1.0, score))  # clip to 0-1

    # Step 6 — Prediction
    prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"
    print(f"Probability of ALERT: {score:.4f}")
    print(f"Prediction: {prediction}")
    return prediction

#-------- TEST WITH REAL SENTENCES --------#
classify_encrypted(input("Enter your prompt"))


In [ ]:
!pip install streamlit
import streamlit as st

In [ ]:
import streamlit as st
import tenseal as ts
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- PAGE CONFIG --------#
st.set_page_config(
    page_title="Vault-LLM",
    page_icon="🔐",
    layout="wide"
)

# -------- LOAD EVERYTHING ONCE --------#
@st.cache_resource
def load_models():
    weights = np.load("vault_weights.npy")
    bias = np.load("vault_bias.npy")
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return weights, bias, embedder, context

weights, bias, embedder, context = load_models()

# -------- PIPELINE --------#
def classify_encrypted(sentence):
    # Step 1 - Embed
    user_vector = embedder.encode(sentence).tolist()

    # Step 2 - Encrypt
    encrypted_vector = ts.ckks_vector(context, user_vector)
    serialized = encrypted_vector.serialize()

    # Step 3 - Server side
    encrypted_dot = encrypted_vector.dot(weights.tolist())
    encrypted_result = encrypted_dot + bias[0]

    # Step 4 - Decrypt
    decrypted_result = encrypted_result.decrypt()[0]

    # Step 5 - Sigmoid
    x = decrypted_result
    score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
    score = max(0.0, min(1.0, score))

    prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"
    return prediction, score, serialized, user_vector

# -------- UI --------#
st.title("🔐 Vault-LLM")
st.subheader("Privacy-Preserving AI Inference Gateway")
st.markdown("*The server classifies your query without ever reading it*")

st.divider()

# Input
sentence = st.text_input(
    "Enter a sentence to classify:",
    placeholder="e.g. Hydraulic pressure critical, shutdown imminent"
)

if st.button("🔒 Encrypt & Classify") and sentence:
    with st.spinner("Encrypting and classifying..."):
        prediction, score, serialized, plain_vector = classify_encrypted(sentence)

    st.divider()

    # Split screen
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### 👤 What You See")
        st.success(f"**Input:** {sentence}")
        st.metric("Prediction", prediction)
        st.metric("Confidence", f"{score:.2%}")
        st.markdown("**Plain vector (first 5 values):**")
        st.code(str([round(x, 4) for x in plain_vector[:5]]))

    with col2:
        st.markdown("### 👾 What The Server Sees")
        st.error("**Input:** [ENCRYPTED]")
        st.metric("Prediction", "???")
        st.metric("Confidence", "???")
        st.markdown("**Encrypted blob (first 50 bytes):**")
        st.code(str(serialized[:50]))

    st.divider()

    # Result banner
    if prediction == "ALERT 🚨":
        st.error(f"## {prediction} — Threat detected with {score:.2%} confidence")
    else:
        st.success(f"## {prediction} — No threat detected ({score:.2%} confidence)")

In [ ]:
%%writefile app.py

In [ ]:
import streamlit as st
import tenseal as ts
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- PAGE CONFIG --------#
st.set_page_config(
    page_title="Vault-LLM",
    page_icon="🔐",
    layout="wide"
)

# -------- LOAD EVERYTHING ONCE --------#
@st.cache_resource
def load_models():
    weights = np.load("vault_weights.npy")
    bias = np.load("vault_bias.npy")
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return weights, bias, embedder, context

weights, bias, embedder, context = load_models()

# -------- PIPELINE --------#
def classify_encrypted(sentence):
    # Step 1 - Embed
    user_vector = embedder.encode(sentence).tolist()

    # Step 2 - Encrypt
    encrypted_vector = ts.ckks_vector(context, user_vector)
    serialized = encrypted_vector.serialize()

    # Step 3 - Server side
    encrypted_dot = encrypted_vector.dot(weights.tolist())
    encrypted_result = encrypted_dot + bias[0]

    # Step 4 - Decrypt
    decrypted_result = encrypted_result.decrypt()[0]

    # Step 5 - Sigmoid
    x = decrypted_result
    score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
    score = max(0.0, min(1.0, score))

    prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"
    return prediction, score, serialized, user_vector

# -------- UI --------#
st.title("🔐 Vault-LLM")
st.subheader("Privacy-Preserving AI Inference Gateway")
st.markdown("*The server classifies your query without ever reading it*")

st.divider()

# Input
sentence = st.text_input(
    "Enter a sentence to classify:",
    placeholder="e.g. Hydraulic pressure critical, shutdown imminent"
)

if st.button("🔒 Encrypt & Classify") and sentence:
    with st.spinner("Encrypting and classifying..."):
        prediction, score, serialized, plain_vector = classify_encrypted(sentence)

    st.divider()

    # Split screen
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### 👤 What You See")
        st.success(f"**Input:** {sentence}")
        st.metric("Prediction", prediction)
        st.metric("Confidence", f"{score:.2%}")
        st.markdown("**Plain vector (first 5 values):**")
        st.code(str([round(x, 4) for x in plain_vector[:5]]))

    with col2:
        st.markdown("### 👾 What The Server Sees")
        st.error("**Input:** [ENCRYPTED]")
        st.metric("Prediction", "???")
        st.metric("Confidence", "???")
        st.markdown("**Encrypted blob (first 50 bytes):**")
        st.code(str(serialized[:50]))

    st.divider()

    # Result banner
    if prediction == "ALERT 🚨":
        st.error(f"## {prediction} — Threat detected with {score:.2%} confidence")
    else:
        st.success(f"## {prediction} — No threat detected ({score:.2%} confidence)")

In [ ]:
from pyngrok import ngrok
import subprocess
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])


In [ ]:
from pyngrok import ngrok
import subprocess

# Paste your token here
ngrok.set_auth_token("3C4bCdJRWUY9cYrnbnVX9AVt6H7_5jitNYFShDmFreMWnAyWk")

subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

In [ ]:
!curl http://localhost:8501

In [ ]:
import subprocess
result = subprocess.run(['streamlit', 'run', 'app.py', '--server.port', '8501'],
                      capture_output=True, text=True, timeout=10)
print(result.stderr)

In [ ]:
from pyngrok import ngrok
import subprocess
import time

ngrok.set_auth_token("3C4bCdJRWUY9cYrnbnVX9AVt6H7_5jitNYFShDmFreMWnAyWk")

subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false'
])

time.sleep(10)
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

In [ ]:
import requests
import time

time.sleep(5)

try:
    response = requests.get("http://localhost:8501")
    print(f"Status code: {response.status_code}")
    print("Streamlit is running ✅")
except Exception as e:
    print(f"Streamlit is NOT running ❌")
    print(f"Error: {e}")

In [ ]:
# Install localtunnel
!npm install -g localtunnel

# Get your external IP (you'll need this)
import requests
ip = requests.get("https://api.ipify.org").text
print(f"Your IP: {ip}")

In [ ]:
import subprocess
import threading

def run_tunnel():
    subprocess.run(['lt', '--port', '8501'])

thread = threading.Thread(target=run_tunnel)
thread.daemon = True
thread.start()

import time
time.sleep(5)
print("Tunnel started — check output above for your URL")

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
import os
for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".npy"):
            print(os.path.join(root, file))

In [ ]:
weights = np.load("vault_weights.npy")  # full path
bias = np.load("vault_bias.npy")         # full path

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'streamlit'])
import time
time.sleep(2)
print("Cleared ✅")

In [ ]:
import subprocess
subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
    '--server.address', '0.0.0.0'
])
import time
time.sleep(8)
print("Streamlit launched ✅")

In [ ]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Open this URL: {url}")

In [ ]:
!pip install gradio tenseal sentence-transformers

In [ ]:
import gradio as gr
import tenseal as ts
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- LOAD EVERYTHING --------#
weights = np.load("/content/vault_weights.npy")
bias = np.load("/content/vault_bias.npy")
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

print("Everything loaded ✅")

# -------- PIPELINE --------#
def classify_encrypted(sentence):
    # Embed
    user_vector = embedder.encode(sentence).tolist()

    # Encrypt
    encrypted_vector = ts.ckks_vector(context, user_vector)
    serialized = encrypted_vector.serialize()

    # Server side
    encrypted_dot = encrypted_vector.dot(weights.tolist())
    encrypted_result = encrypted_dot + bias[0]

    # Decrypt
    decrypted_result = encrypted_result.decrypt()[0]

    # Sigmoid
    x = decrypted_result
    score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
    score = max(0.0, min(1.0, score))

    prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"

    # What server sees
    server_view = f"Encrypted blob (first 30 bytes):\n{serialized[:30]}\n\nPrediction: ???\nConfidence: ???"

    # What user sees
    user_view = f"Prediction: {prediction}\nConfidence: {score:.2%}\nVector preview: {[round(x,4) for x in user_vector[:5]]}"

    return user_view, server_view

# -------- UI --------#
with gr.Blocks(title="🔐 Vault-LLM") as app:
    gr.Markdown("# 🔐 Vault-LLM")
    gr.Markdown("### Privacy-Preserving AI Inference Gateway")
    gr.Markdown("*The server classifies your query without ever reading it*")

    with gr.Row():
        sentence_input = gr.Textbox(
            label="Enter your sentence",
            placeholder="e.g. Hydraulic pressure critical, shutdown imminent"
        )

    classify_btn = gr.Button("🔒 Encrypt & Classify", variant="primary")

    with gr.Row():
        user_output = gr.Textbox(label="👤 What You See", lines=6)
        server_output = gr.Textbox(label="👾 What The Server Sees", lines=6)

    classify_btn.click(
        fn=classify_encrypted,
        inputs=sentence_input,
        outputs=[user_output, server_output]
    )

app.launch(share=True)